# 10.2 长上下文注意力 (Long Context Attention)

处理长序列时标准注意力的O(n²)复杂度成为瓶颈，需要高效的注意力变体。

本节涵盖：
- 稀疏注意力
- 线性注意力
- 状态空间模型（SSM/Mamba）

## 1. 稀疏注意力

**目的**：减少长序列注意力的计算量

**基本原理**：只计算部分token对之间的注意力，跳过不重要的注意力连接，将复杂度从O(n²)降至O(n√n)或更低。

**典型模式**：
- **局部窗口**：每个token只关注附近的w个token
- **全局token**：少数特殊token关注所有位置
- **扩张注意力**：按固定间隔采样注意力位置

In [ ]:
import torch
import torch.nn.functional as F
import math

torch.manual_seed(42)

seq_len = 64
window_size = 8

full_mask = torch.ones(seq_len, seq_len)
local_mask = torch.zeros(seq_len, seq_len)
for i in range(seq_len):
    start = max(0, i - window_size // 2)
    end = min(seq_len, i + window_size // 2 + 1)
    local_mask[i, start:end] = 1

causal_mask = torch.tril(torch.ones(seq_len, seq_len))
sparse_mask = local_mask * causal_mask

full_n = (causal_mask == 1).sum().item()
sparse_n = (sparse_mask == 1).sum().item()

print('=== Sparse Attention ===')
print(f'Sequence length: {seq_len}, Window size: {window_size}')
print(f'\nFull causal attention: {full_n} connections')
print(f'Sparse (local window): {sparse_n} connections ({sparse_n/full_n:.1%})')
print(f'\nComplexity comparison:')
for n in [1024, 4096, 16384, 65536]:
    full_ops = n * n
    sparse_ops = n * window_size
    print(f'  n={n:>6d}: full={full_ops/1e9:.1f}G, sparse={sparse_ops/1e6:.1f}M ({sparse_ops/full_ops:.1%})')
print(f'\nKey: Sparse attention reduces computation from O(n²) to O(n*w).')

## 2. 线性注意力与SSM/Mamba

**线性注意力**：用核函数近似softmax，将注意力复杂度降至O(n)，但表达能力受限。

**SSM/Mamba**：用状态空间模型替代注意力，通过递归计算实现O(n)复杂度，同时保持强建模能力。

**Mamba核心**：
- 选择性状态空间：根据输入动态调整状态转移矩阵
- 硬件感知算法：扫描而非卷积，高效利用GPU
- 线性复杂度：推理时每步O(1)，训练时O(n)

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(42)

class SimpleSSM(nn.Module):
    def __init__(self, d_model=64, d_state=16):
        super().__init__()
        self.d_model = d_model
        self.d_state = d_state
        self.A = nn.Parameter(torch.randn(d_state, d_state) * 0.01)
        self.B = nn.Parameter(torch.randn(d_model, d_state) * 0.01)
        self.C = nn.Parameter(torch.randn(d_state, d_model) * 0.01)
        self.D = nn.Parameter(torch.ones(d_model))
        self.dt_proj = nn.Linear(d_model, d_state)

    def forward(self, x):
        B, T, D = x.shape
        h = torch.zeros(B, self.d_state, device=x.device)
        outputs = []
        for t in range(T):
            x_t = x[:, t, :]
            dt = torch.sigmoid(self.dt_proj(x_t)).unsqueeze(-1)
            A_t = torch.exp(self.A * dt)
            B_t = (x_t @ self.B).unsqueeze(-1)
            h = A_t.transpose(-2, -1) @ h.unsqueeze(-1) + B_t * x_t.unsqueeze(1).transpose(-2, -1)
            h = h.squeeze(-1)
            y_t = (h @ self.C) + x_t * self.D
            outputs.append(y_t)
        return torch.stack(outputs, dim=1)

ssm = SimpleSSM(64, 16)
x = torch.randn(2, 32, 64)
out = ssm(x)

print('=== SSM/Mamba ===')
print(f'Input: {x.shape}, Output: {out.shape}')
print(f'Parameters: {sum(p.numel() for p in ssm.parameters()):,}')
print(f'\nComplexity comparison:')
print(f'  Attention: O(n² × d) - quadratic in sequence length')
print(f'  SSM/Mamba: O(n × d × d_state) - linear in sequence length')
print(f'\nKey: Mamba achieves linear complexity with selective state updates.')
print(f'Combined with attention in Jamba for best of both worlds.')